> **Corrections apportées lors de l'intégration**
> - Les identifiants (`PAT-…`, `FAC-…`, `LIC-…`) gardent leur écriture d'origine, pour permettre les jointures avec les autres domaines.
> - Les mesures textuelles (`text_value`) et les consultations sans date de suivi sont conservées : leur absence de valeur est normale.
> - Les valeurs avec unité collée (`82.2 index`) sont converties en nombres au lieu de devenir vides.
> - Le fichier `clinical_procedures` est désormais nettoyé.
> - Les tables sont enregistrées dans `data_clean/health/`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings as wr
import statistics
from scipy import stats
import re
from pathlib import Path

wr.filterwarnings('ignore')

In [2]:
df_cardiac_recording=pd.read_csv('./data/health/cardiac_recordings.csv')
df_cardiac_signal_samples=pd.read_csv('./data/health/cardiac_signal_samples.csv')
df_clinical_encounters=pd.read_csv('./data/health/clinical_encounters.csv')
df_cardiac_notes=pd.read_csv('./data/health/clinical_notes.csv')
df_clinical_observations=pd.read_csv('./data/health/clinical_observations.csv')
df_patient_administration=pd.read_csv('./data/health/patient_administration.csv')
df_clinical_procedures=pd.read_csv('./data/health/clinical_procedures.csv')


## **Fonctions**

In [3]:
# Stats
# permettant de voir si des colonnes objet serait un int caché
def get_dtypes_report(df):
    """Types de chaque colonne + signale les colonnes 'object' qui ressemblent à des nombres."""
    report = pd.DataFrame({"dtype": df.dtypes})
    def looks_numeric(s):
        if s.dtype != "object":
            return False
        converted = pd.to_numeric(s.dropna(), errors="coerce")
        return len(converted) > 0 and converted.notna().mean() > 0.9
    report["looks_numeric"] = [looks_numeric(df[c]) for c in df.columns]
    return report

# Dans describe(), seule la std est affichée. Pour la comparer à la variance
def variance_std(dataset):
    variance_list=[]
    ecartètype_list=[]
    data= dataset.select_dtypes(exclude=['object'])
    mu = np.mean(data)
    print('Variance & std :')
    for column in data:
        mu = np.mean(data[column])
        variance_column = int(np.sum((data[column] - mu) ** 2) / len(data[column]))
        variance_list.append(variance_column)
        
        ecart_type = np.std(data[column])
        ecartètype_list.append(ecart_type)
        
        print(f'{column} : variance : {variance_column}, std : {round(ecart_type, 2)}')
    return variance_list, ecartètype_list


def get_iqr_bounds(df, columns=None, factor=1.5):
    """Calcule Q1, Q3, IQR et les bornes de détection pour chaque colonne numérique."""
    cols = columns or df.select_dtypes(include="number").columns
    num = df[cols]
    q1, q3 = num.quantile(0.25), num.quantile(0.75)
    iqr = q3 - q1
    return pd.DataFrame({
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower": q1 - factor * iqr,
        "upper": q3 + factor * iqr,
    })

In [4]:
# Cleaning 

# retire les espace en début/fin de string et retire les espaces multiples
def strip_whitespace(df, cols=None):
    df = df.copy()
    cols = cols if cols is not None else df.select_dtypes(include=["object", "string"]).columns
    for c in cols:
        s = df[c].astype("string")
        df[c] = s.str.strip().str.replace(r"\s+", " ", regex=True)
    return df

# Normalise en lower 
def normalize_case(df, cols, mode="lower"):
    modes = {"lower": "lower", "upper": "upper", "title": "title"}
    if mode not in modes:
        raise ValueError(f"mode doit être parmi {list(modes)}")
    df = df.copy()
    for c in cols:
        df[c] = getattr(df[c].astype("string").str, modes[mode])()
    return df

# Retire les colonnes avec trop de valeurs manquantes (>15%) sauf celles dans exclude)
def drop_high_missing_columns(df, threshold=0.15, exclude=None, verbose=True):
    exclude = set(exclude or [])
    ratio = df.isna().mean()
    to_drop = [c for c in ratio[ratio > threshold].index if c not in exclude]

    if verbose:
        for c in to_drop:
            print(f"drop_high_missing_columns : '{c}' supprimée ({ratio[c]:.1%} manquants)")
        if not to_drop:
            print("drop_high_missing_columns : aucune colonne supprimée")

    return df.drop(columns=to_drop)

# Normalise le format de date. Si aucun format, la valeur devient NaT 
def parse_dates(df, cols, formats, errors="coerce", dayfirst=False):
    df = df.copy()
    for c in cols:
        s = df[c].astype("string").str.strip()
        result = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
        remaining = s.notna()
        for fmt in formats:
            if not remaining.any():
                break
            parsed = pd.to_datetime(s[remaining], format=fmt, errors="coerce")
            ok = parsed.notna()
            result.loc[parsed.index[ok]] = parsed[ok]
            remaining.loc[parsed.index[ok]] = False
        if remaining.any() and errors == "raise":
            bad = s[remaining].unique()[:5]
            raise ValueError(f"Colonne '{c}' : formats non reconnus, ex. {list(bad)}")
        df[c] = result
    return df

# retire les doublons strictement identiques
def remove_exact_duplicates(df, subset=None, keep="first", verbose=True):
    before = len(df)
    out = df.drop_duplicates(subset=subset, keep=keep).reset_index(drop=True)
    if verbose:
        print(f"remove_exact_duplicates : {before - len(out)} ligne(s) supprimée(s)")
    return out

# Vérifie l'unicité d'une clé
def check_unique_key(df, key, raise_error=False):
    key = [key] if isinstance(key, str) else list(key)
    dup = df[df.duplicated(subset=key, keep=False)].sort_values(key)
    if not dup.empty:
        msg = f"Clé {key} non unique : {dup[key].drop_duplicates().shape[0]} valeur(s) en doublon. Mais dont les valeur sont différentes (vérifié par la fonction remove_exact_duplicates)"
        if raise_error:
            raise ValueError(msg)
        print(msg)
    return dup

# Rapport rapide sur les données nettoyées du dataset
def quality_report(df, name="dataset"):
    """Rapport synthétique : dimensions, doublons, manquants, constantes, types."""
    n_unique = df.nunique(dropna=False)
    report = {
        "name": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "cells_missing": int(df.isna().sum().sum()),
        "pct_cells_missing": round(df.isna().mean().mean() * 100, 2),
        "cols_with_missing": df.columns[df.isna().any()].tolist(),
        "constant_cols": n_unique[n_unique <= 1].index.tolist(),
    }
    return report


# Colonnes de texte à mettre en minuscules.
# On garde telles quelles les colonnes d'identifiants, reconnaissables à leurs valeurs
# (PAT-31162, FAC-VLN-101, LIC-VLN-MED-1001…) : elles doivent rester identiques à celles
# des autres domaines pour permettre les jointures.
# On garde aussi les colonnes passées dans `exclure` (noms de personnes, textes longs, codes de mesure).
def colonnes_categories(df, exclure=()):
    texte = df.select_dtypes(include=["object", "string"]).columns
    garder = []
    for c in texte:
        valeurs = df[c].dropna().astype("string").str.strip()
        ressemble_identifiant = valeurs.str.match(r"^[A-Za-z]+-[A-Za-z0-9-]*\d").mean() > 0.5
        if c not in exclure and not ressemble_identifiant:
            garder.append(c)
    return garder

# Convertit une colonne de texte en nombres.
# Gère la virgule décimale (« 84,9 ») et l'unité collée à la valeur (« 82.2 index »).
def to_number(serie):
    texte = serie.astype("string").str.strip().str.replace(",", ".", regex=False)
    nombre = texte.str.extract(r"^(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(nombre, errors="coerce")


## **1/6 Cardiac_recordings**

In [5]:
df_cardiac_recording.head()

,recording_id,patient_number,encounter_number,recording_date,recording_context,facility_license_number,reviewing_license_number,device_serial_number,device_model,sampling_rate_hz,sampling_interval_ms,duration_seconds,channel_count,sample_count,amplitude_unit,automated_interpretation,recording_notes_code,device_quality,archive_status
0,CRD-410031,PAT-31162,ENC-000004,2022-06-21,rest,FAC-VLN-101,LIC-VLN-RES-1016,DEV-CARD-07034,Vantage Rhythm Monitor,100,10,12.0,2,1200,normalized_unit,complex,rhythm_variability,80.9,finalized
1,CRD-410019,PAT-31162,ENC-000004,2022-06-21,technical_repeat,FAC-VLN-101,LIC-VLN-RES-1016,DEV-CARD-07044,HelixCard Duo,100,10,30.0,2,3000,normalized_unit,irregular,repeat_recommended,88.0,finalized
2,CRD-410025,PAT-31040,ENC-000007,2017-03-16,rest,FAC-AVL-201,LIC-VLN-MED-1003,DEV-CARD-07027,Asterion CV-3,125,8,12.0,2,1500,normalized_unit,review_required,no_note,84.8,finalized
3,CRD-410016,PAT-31115,ENC-000017,2025-09-11,rest,FAC-AVL-301,LIC-VLN-MED-1006,DEV-CARD-07019,Asterion CV-2,125,8,30.0,2,3750,normalized_unit,inconclusive,signal_overlap,86.7,finalized
4,CRD-410014,PAT-31233,ENC-000033,2023-04-25,rest,FAC-AVL-301,LIC-VLN-MED-1001,DEV-CARD-07022,HelixCard Duo,125,8,18.0,2,2250,normalized_unit,complex,rhythm_variability,93.0,finalized


## Stats Cardiac_recordings

In [6]:
df_cardiac_recording.describe()

,sampling_rate_hz,sampling_interval_ms,duration_seconds,channel_count,sample_count,device_quality
count,44.000000,44.000000,44.000000,44.0,44.000000,44.000000
mean,112.500000,9.000000,19.545455,2.0,2202.840909,91.368182
std,12.644513,1.011561,5.990126,0.0,729.790029,5.132059
min,100.000000,8.000000,12.000000,2.0,1200.000000,76.900000
25%,100.000000,8.000000,15.000000,2.0,1500.000000,87.850000
50%,112.500000,9.000000,18.000000,2.0,2000.000000,92.550000
75%,125.000000,10.000000,24.000000,2.0,3000.000000,95.600000
max,125.000000,10.000000,30.000000,2.0,3750.000000,98.000000


In [7]:
# compte le nombre de valeurs uniques par colonne
unique_cols_Cardiac_recordings = []
for column in df_cardiac_recording.columns:
    print(f'{column} : {df_cardiac_recording[column].nunique()}')
    if df_cardiac_recording[column].nunique() == 1:
        unique_cols_Cardiac_recordings.append(column)

print('---')
print(f'colonnes avec une seule valeur : {unique_cols_Cardiac_recordings}')


recording_id : 44
patient_number : 42
encounter_number : 42
recording_date : 42
recording_context : 5
facility_license_number : 3
reviewing_license_number : 8
device_serial_number : 44
device_model : 4
sampling_rate_hz : 2
sampling_interval_ms : 2
duration_seconds : 6
channel_count : 1
sample_count : 10
amplitude_unit : 1
automated_interpretation : 6
recording_notes_code : 6
device_quality : 39
archive_status : 1
---
colonnes avec une seule valeur : ['channel_count', 'amplitude_unit', 'archive_status']


In [8]:
print('Aucune valeur manquante')
df_cardiac_recording.isna().sum()

Aucune valeur manquante


recording_id                0
patient_number              0
encounter_number            0
recording_date              0
recording_context           0
facility_license_number     0
reviewing_license_number    0
device_serial_number        0
device_model                0
sampling_rate_hz            0
sampling_interval_ms        0
duration_seconds            0
channel_count               0
sample_count                0
amplitude_unit              0
automated_interpretation    0
recording_notes_code        0
device_quality              0
archive_status              0
dtype: int64

In [9]:
print('Aucun int caché')
get_dtypes_report(df_cardiac_recording)

Aucun int caché


,dtype,looks_numeric
recording_id,str,False
patient_number,str,False
encounter_number,str,False
recording_date,str,False
recording_context,str,False
facility_license_number,str,False
reviewing_license_number,str,False
device_serial_number,str,False
device_model,str,False
sampling_rate_hz,int64,False


In [10]:
print(variance_std(df_cardiac_recording))

Variance & std :
sampling_rate_hz : variance : 156, std : 12.5
sampling_interval_ms : variance : 1, std : 1.0
duration_seconds : variance : 35, std : 5.92
channel_count : variance : 0, std : 0.0
sample_count : variance : 520489, std : 721.45
device_quality : variance : 25, std : 5.07
([156, 1, 35, 0, 520489, 25], [np.float64(12.5), np.float64(1.0), np.float64(5.9216649434495485), np.float64(0.0), np.float64(721.4492971279731), np.float64(5.07340537989626)])


In [11]:
print('forte correlation entre sample_count et duration_seconds')
df_cardiac_recording.corr(numeric_only=True)

forte correlation entre sample_count et duration_seconds


,sampling_rate_hz,sampling_interval_ms,duration_seconds,channel_count,sample_count,device_quality
sampling_rate_hz,1.000000,-1.000000,0.053732,NaN,0.388265,0.002688
sampling_interval_ms,-1.000000,1.000000,-0.053732,NaN,-0.388265,-0.002688
duration_seconds,0.053732,-0.053732,1.000000,NaN,0.935395,-0.252317
channel_count,NaN,NaN,NaN,NaN,NaN,NaN
sample_count,0.388265,-0.388265,0.935395,NaN,1.000000,-0.223649
device_quality,0.002688,-0.002688,-0.252317,NaN,-0.223649,1.000000


In [12]:
# Pour rappel : 
# q1 : 1er quartile, la valeur sous laquelle se trouvent 25 % des données
# q3 : 3e quartile, la valeur sous laquelle se trouvent 75 % des données
# iqr : écart interquartile, la différence entre q3 et q1
# lower : borne basse, q1 - 1.5 × iqr. Une valeur en dessous est un outlier
# upper : borne haute, q3 + 1.5 × iqr. Une valeur au-dessus est un outlier
get_iqr_bounds(df_cardiac_recording)

,q1,q3,iqr,lower,upper
sampling_rate_hz,100.00,125.0,25.00,62.500,162.500
sampling_interval_ms,8.00,10.0,2.00,5.000,13.000
duration_seconds,15.00,24.0,9.00,1.500,37.500
channel_count,2.00,2.0,0.00,2.000,2.000
sample_count,1500.00,3000.0,1500.00,-750.000,5250.000
device_quality,87.85,95.6,7.75,76.225,107.225


## Cleaning Cardiac_recordings

- Suppression de 3 colonnes ne comportant qu'une seule valeur : 'channel_count', 'amplitude_unit', 'archive_status'

- Supression des espaces superflux

- Mise en forme de la date (recording_date)

- Suppression des lignes strictement doubles


In [13]:
df_cardiac_recording.drop(unique_cols_Cardiac_recordings, axis=1, inplace=True)

In [14]:
enc = df_cardiac_recording.copy()

print(quality_report(enc, "cardiac_recordings"))

text_cols = colonnes_categories(enc)

df_cardiac_recording_clean = (
    enc
    .pipe(strip_whitespace)
    .pipe(normalize_case, cols=text_cols, mode="lower")
    .pipe(parse_dates,
          cols=['recording_date'],
          formats=["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S",
                   "%d-%b-%Y %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%b-%Y"])
    .pipe(remove_exact_duplicates)
)

# La clé de ce fichier est recording_id (un patient peut avoir plusieurs enregistrements)
check_unique_key(df_cardiac_recording_clean, "recording_id")

{'name': 'cardiac_recordings', 'rows': 44, 'columns': 16, 'duplicate_rows': 0, 'cells_missing': 0, 'pct_cells_missing': np.float64(0.0), 'cols_with_missing': [], 'constant_cols': []}
remove_exact_duplicates : 0 ligne(s) supprimée(s)


,recording_id,patient_number,encounter_number,recording_date,recording_context,facility_license_number,reviewing_license_number,device_serial_number,device_model,sampling_rate_hz,sampling_interval_ms,duration_seconds,sample_count,automated_interpretation,recording_notes_code,device_quality


## **2/6 df_cardiac_signal_samples**

In [15]:
df_cardiac_signal_samples.head()

,recording_id,time_ms,channel_a_normalized,channel_b_normalized
0,CRD-410031,0,-0.118963,-0.200416
1,CRD-410031,10,-0.316781,-0.333479
2,CRD-410031,20,-0.182477,-0.300709
3,CRD-410031,30,-0.017428,-0.507620
4,CRD-410031,40,-0.225540,-0.148937


## Stats df_cardiac_signal_samples

In [16]:
df_cardiac_signal_samples.describe()

,time_ms,channel_a_normalized,channel_b_normalized
count,96925.000000,96925.000000,96925.000000
mean,10675.341759,0.081485,0.052849
std,7084.243888,0.467100,0.415823
min,0.000000,-1.485359,-1.587614
25%,4890.000000,-0.184500,-0.184217
50%,9784.000000,0.002108,-0.002069
75%,15360.000000,0.240280,0.210327
max,29992.000000,3.169961,2.510826


In [17]:
# compte le nombre de valeurs uniques par colonne
unique_cols_Cardiac_recordings = []
for column in df_cardiac_signal_samples.columns:
    print(f'{column} : {df_cardiac_signal_samples[column].nunique()}')
    if df_cardiac_signal_samples[column].nunique() == 1:
        unique_cols_Cardiac_recordings.append(column)

print('---')
print(f'colonnes avec une seule valeur : {unique_cols_Cardiac_recordings}')

recording_id : 44
time_ms : 6000
channel_a_normalized : 93010
channel_b_normalized : 92799
---
colonnes avec une seule valeur : []


In [18]:
print('Aucune valeur manquante')
df_cardiac_signal_samples.isna().sum()

Aucune valeur manquante


recording_id            0
time_ms                 0
channel_a_normalized    0
channel_b_normalized    0
dtype: int64

In [19]:
print('Aucun int caché')
get_dtypes_report(df_cardiac_signal_samples)

Aucun int caché


,dtype,looks_numeric
recording_id,str,False
time_ms,int64,False
channel_a_normalized,float64,False
channel_b_normalized,float64,False


In [20]:
variance_std(df_cardiac_signal_samples)

Variance & std :
time_ms : variance : 50185993, std : 7084.21


channel_a_normalized : variance : 0, std : 0.47
channel_b_normalized : variance : 0, std : 0.42


([50185993, 0, 0],
 [np.float64(7084.207343367341),
  np.float64(0.46709747695232695),
  np.float64(0.4158211340356705)])

In [21]:
# Pour rappel : 
# q1 : 1er quartile, la valeur sous laquelle se trouvent 25 % des données
# q3 : 3e quartile, la valeur sous laquelle se trouvent 75 % des données
# iqr : écart interquartile, la différence entre q3 et q1
# lower : borne basse, q1 - 1.5 × iqr. Une valeur en dessous est un outlier
# upper : borne haute, q3 + 1.5 × iqr. Une valeur au-dessus est un outlier
get_iqr_bounds(df_cardiac_signal_samples)

,q1,q3,iqr,lower,upper
time_ms,4890.000000,15360.000000,10470.000000,-10815.000000,31065.000000
channel_a_normalized,-0.184500,0.240280,0.424780,-0.821670,0.877450
channel_b_normalized,-0.184217,0.210327,0.394544,-0.776033,0.802143


## Cleaning df_cardiac_signal_samples

- Rien de particulier sur les stats. Nettoyage par acquis de concience.

In [22]:
enc = df_cardiac_signal_samples.copy()

print(quality_report(enc, "cardiac_signal_samples"))

df_cardiac_signal_samples_clean = (
    enc
    .pipe(strip_whitespace)
    .pipe(remove_exact_duplicates)
)

{'name': 'cardiac_signal_samples', 'rows': 96925, 'columns': 4, 'duplicate_rows': 0, 'cells_missing': 0, 'pct_cells_missing': np.float64(0.0), 'cols_with_missing': [], 'constant_cols': []}
remove_exact_duplicates : 0 ligne(s) supprimée(s)


In [23]:
df_cardiac_signal_samples_clean.head()

,recording_id,time_ms,channel_a_normalized,channel_b_normalized
0,CRD-410031,0,-0.118963,-0.200416
1,CRD-410031,10,-0.316781,-0.333479
2,CRD-410031,20,-0.182477,-0.300709
3,CRD-410031,30,-0.017428,-0.507620
4,CRD-410031,40,-0.225540,-0.148937


## **3/6 df_clinical_encounters**

In [24]:
df_clinical_encounters.head()

,encounter_number,patient_number,encounter_start,encounter_end,facility_license_number,encounter_type,referral_source,attending_license_number,service_unit,disposition,confidentiality_class,follow_up_date
0,ENC-000001,PAT-31162,2018-02-27T09:15:00,2018-02-27T10:45:00,FAC-VLN-101,mobility_assessment,research programme,LIC-VLN-MED-1001,cardiac diagnostics,follow_up_planned,standard,2019-09-04
1,ENC-000002,PAT-31162,2019-09-04T10:15:00,2019-09-04T11:20:00,FAC-VLN-101,mobility_assessment,self,LIC-VLN-MED-1003,cardiac diagnostics,follow_up_planned,standard,2021-01-14
2,ENC-000003,PAT-31162,2021-01-14T11:15:00,2021-01-14T12:33:00,FAC-VLN-101,cardiac_assessment,executive health office,LIC-VLN-RES-1016,general assessment,follow_up_planned,confidential,2022-06-21
3,ENC-000004,PAT-31162,2022-06-21T12:15:00,2022-06-21T13:38:00,FAC-VLN-101,cardiac_assessment,primary care,LIC-VLN-RES-1016,general assessment,follow_up_planned,confidential,2023-11-18
4,ENC-000005,PAT-31162,2023-11-18T13:15:00,2023-11-18T14:08:00,FAC-VLN-101,mobility_assessment,occupational service,LIC-VLN-MED-1002,general assessment,follow_up_planned,confidential,2025-05-17


In [25]:
df_clinical_encounters.describe()

,encounter_number,patient_number,encounter_start,encounter_end,facility_license_number,encounter_type,referral_source,attending_license_number,service_unit,disposition,confidentiality_class,follow_up_date
count,1069,1069,1069,1069,1069,1069,1066,1069,1069,1069,1069,829
unique,1067,240,998,1065,3,15,8,8,8,7,3,673
top,ENC-000733,PAT-31221,2025-06-20T13:15:00,2021-10-31T10:14:00,FAC-VLN-101,specialist_follow_up,executive health office,LIC-VLN-MED-1002,cardiac diagnostics,follow_up_planned,standard,2025-06-20
freq,2,11,17,2,659,243,226,162,279,807,583,22


In [26]:
# compte le nombre de valeurs uniques par colonne
unique_cols_Cardiac_recordings = []
for column in df_clinical_encounters.columns:
    print(f'{column} : {df_clinical_encounters[column].nunique()}')
    if df_clinical_encounters[column].nunique() == 1:
        unique_cols_Cardiac_recordings.append(column)
        
print('---')
print(f'colonnes avec une seule valeur : {unique_cols_Cardiac_recordings}')

encounter_number : 1067
patient_number : 240
encounter_start : 998
encounter_end : 1065
facility_license_number : 3
encounter_type : 15
referral_source : 8
attending_license_number : 8
service_unit : 8
disposition : 7
confidentiality_class : 3
follow_up_date : 673
---
colonnes avec une seule valeur : []


In [27]:
print('Valeurs manquantes sur follow_up_date')
df_clinical_encounters.isna().sum()

Valeurs manquantes sur follow_up_date


encounter_number              0
patient_number                0
encounter_start               0
encounter_end                 0
facility_license_number       0
encounter_type                0
referral_source               3
attending_license_number      0
service_unit                  0
disposition                   0
confidentiality_class         0
follow_up_date              240
dtype: int64

In [28]:
print('Aucun int caché')
get_dtypes_report(df_clinical_encounters)


Aucun int caché


,dtype,looks_numeric
encounter_number,str,False
patient_number,str,False
encounter_start,str,False
encounter_end,str,False
facility_license_number,str,False
encounter_type,str,False
referral_source,str,False
attending_license_number,str,False
service_unit,str,False
disposition,str,False


## Cleaning Clinical_encounters

- Suppression des 240 valeurs manquantes pour follow_up_date

- Stadarisation et supression du bruit dans les dates de encounter_start,  encounter_end, follow_up_date

- Normalisation des string en minuscule


In [29]:
enc = df_clinical_encounters.copy()

print(quality_report(enc, "clinical_encounters"))

text_cols = colonnes_categories(enc)

df_clinical_encounters_clean = (
    enc
    .pipe(strip_whitespace)
    .pipe(normalize_case, cols=text_cols, mode="lower")
    .pipe(parse_dates,
          cols=['encounter_start', 'encounter_end', 'follow_up_date'],
          formats=["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S",
                   "%d-%b-%Y %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%b-%Y"])
    .pipe(remove_exact_duplicates)
)

# Les consultations sans follow_up_date sont conservées :
# une date absente signifie qu'aucun suivi n'est prévu, pas une erreur.
print("Consultations sans date de suivi :", df_clinical_encounters_clean["follow_up_date"].isna().sum())

{'name': 'clinical_encounters', 'rows': 1069, 'columns': 12, 'duplicate_rows': 2, 'cells_missing': 243, 'pct_cells_missing': np.float64(1.89), 'cols_with_missing': ['referral_source', 'follow_up_date'], 'constant_cols': []}


remove_exact_duplicates : 2 ligne(s) supprimée(s)
Consultations sans date de suivi : 240


In [30]:
df_clinical_encounters_clean.head()

,encounter_number,patient_number,encounter_start,encounter_end,facility_license_number,encounter_type,referral_source,attending_license_number,service_unit,disposition,confidentiality_class,follow_up_date
0,ENC-000001,PAT-31162,2018-02-27 09:15:00,2018-02-27 10:45:00,FAC-VLN-101,mobility_assessment,research programme,LIC-VLN-MED-1001,cardiac diagnostics,follow_up_planned,standard,2019-09-04
1,ENC-000002,PAT-31162,2019-09-04 10:15:00,2019-09-04 11:20:00,FAC-VLN-101,mobility_assessment,self,LIC-VLN-MED-1003,cardiac diagnostics,follow_up_planned,standard,2021-01-14
2,ENC-000003,PAT-31162,2021-01-14 11:15:00,2021-01-14 12:33:00,FAC-VLN-101,cardiac_assessment,executive health office,LIC-VLN-RES-1016,general assessment,follow_up_planned,confidential,2022-06-21
3,ENC-000004,PAT-31162,2022-06-21 12:15:00,2022-06-21 13:38:00,FAC-VLN-101,cardiac_assessment,primary care,LIC-VLN-RES-1016,general assessment,follow_up_planned,confidential,2023-11-18
4,ENC-000005,PAT-31162,2023-11-18 13:15:00,2023-11-18 14:08:00,FAC-VLN-101,mobility_assessment,occupational service,LIC-VLN-MED-1002,general assessment,follow_up_planned,confidential,2025-05-17


## **4/6 Cardiac_notes**

## Stats

In [31]:
df_cardiac_notes.head()

,document_id,patient_number,encounter_number,document_date,document_type,author_license_number,facility_license_number,title,text,confidentiality_class,superseded_document_id
0,CLN-510027,PAT-31065,ENC-000037,2023-04-11,imaging_report,LIC-VLN-MED-1003,FAC-VLN-101,Targeted Thoracic Imaging Review,INDICATION\nReview of recurrent discordance be...,restricted,NaN
1,CLN-510055,PAT-31115,ENC-000016,2024-01-19,specialist_consultation,LIC-VLN-MED-1002,FAC-AVL-301,Consultation on Recurrent Capture Discrepancies,REASON FOR CONSULTATION\nAssessment of repeate...,restricted,NaN
2,CLN-510025,PAT-31065,ENC-000038,2025-01-24,specialist_consultation,LIC-VLN-MED-1003,FAC-VLN-101,Restricted Phenotype and Public-Identity Resol...,REASON FOR REVIEW\nRestricted-record consultat...,restricted,NaN
3,CLN-510064,PAT-31012,ENC-000044,2024-10-03,follow_up_note,LIC-VLN-RES-1044,FAC-AVL-201,Persistent Multisystem Follow-up,INTERVAL HISTORY\nThe patient returned after r...,confidential,NaN
4,CLN-510107,PAT-31115,ENC-000014,2021-06-08,clinical_summary,LIC-VLN-MED-1004,FAC-AVL-301,Clinical Summary 01,CONTEXT\nClinical summary for patient PAT-3111...,confidential,NaN


In [32]:
df_cardiac_notes.describe()


,superseded_document_id
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [33]:
# compte le nombre de valeurs uniques par colonne
unique_cols_Cardiac_notes = []
for column in df_cardiac_notes.columns:
    print(f'{column} : {df_cardiac_notes[column].nunique()}')
    if df_cardiac_notes[column].nunique() == 1:
        unique_cols_Cardiac_notes.append(column)
print('---')
print(f'colonnes avec une seule valeur : {unique_cols_Cardiac_notes}')

document_id : 30
patient_number : 27
encounter_number : 30
document_date : 30
document_type : 5
author_license_number : 8
facility_license_number : 3
title : 30
text : 30
confidentiality_class : 3
superseded_document_id : 0
---
colonnes avec une seule valeur : []


In [34]:
print('Aucune valeur manquante')
df_cardiac_notes.isna().sum()

Aucune valeur manquante


document_id                 0
patient_number              0
encounter_number            0
document_date               0
document_type               0
author_license_number       0
facility_license_number     0
title                       0
text                        0
confidentiality_class       0
superseded_document_id     30
dtype: int64

In [35]:
print('Aucun int caché')
get_dtypes_report(df_cardiac_notes)

Aucun int caché


,dtype,looks_numeric
document_id,str,False
patient_number,str,False
encounter_number,str,False
document_date,str,False
document_type,str,False
author_license_number,str,False
facility_license_number,str,False
title,str,False
text,str,False
confidentiality_class,str,False


In [36]:
print(variance_std(df_cardiac_notes))


Variance & std :
superseded_document_id : variance : 0, std : nan
([0], [np.float64(nan)])


## Cleaning

In [37]:
enc = df_cardiac_notes.copy()

print(quality_report(enc, "clinical_notes"))

# Le texte des notes garde sa casse et ses retours à la ligne (sections INDICATION, FINDINGS…),
# utiles pour l'analyse textuelle. On retire seulement les espaces superflus.
enc["text"] = (enc["text"].astype("string")
               .str.replace("\r\n", "\n", regex=False)
               .str.replace(r"[ \t]+", " ", regex=True)
               .str.strip())

autres = [c for c in enc.select_dtypes(include=["object", "string"]).columns if c != "text"]
text_cols = colonnes_categories(enc, exclure=["text", "title"])

df_cardiac_notes_clean = (
    enc
    .pipe(strip_whitespace, cols=autres)
    .pipe(normalize_case, cols=text_cols, mode="lower")
    .pipe(parse_dates,
          cols=['document_date'],
          formats=["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S",
                   "%d-%b-%Y %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%b-%Y"])
    .pipe(remove_exact_duplicates)
)

# superseded_document_id est vide pour les 30 notes : aucune note n'en remplace une autre.
# La colonne n'apporte aucune information, on la retire.
df_cardiac_notes_clean = df_cardiac_notes_clean.drop(columns=["superseded_document_id"])

{'name': 'clinical_notes', 'rows': 30, 'columns': 11, 'duplicate_rows': 0, 'cells_missing': 30, 'pct_cells_missing': np.float64(9.09), 'cols_with_missing': ['superseded_document_id'], 'constant_cols': ['superseded_document_id']}
remove_exact_duplicates : 0 ligne(s) supprimée(s)


In [38]:
df_cardiac_notes_clean.head()

,document_id,patient_number,encounter_number,document_date,document_type,author_license_number,facility_license_number,title,text,confidentiality_class
0,CLN-510027,PAT-31065,ENC-000037,2023-04-11,imaging_report,LIC-VLN-MED-1003,FAC-VLN-101,Targeted Thoracic Imaging Review,INDICATION\nReview of recurrent discordance be...,restricted
1,CLN-510055,PAT-31115,ENC-000016,2024-01-19,specialist_consultation,LIC-VLN-MED-1002,FAC-AVL-301,Consultation on Recurrent Capture Discrepancies,REASON FOR CONSULTATION\nAssessment of repeate...,restricted
2,CLN-510025,PAT-31065,ENC-000038,2025-01-24,specialist_consultation,LIC-VLN-MED-1003,FAC-VLN-101,Restricted Phenotype and Public-Identity Resol...,REASON FOR REVIEW\nRestricted-record consultat...,restricted
3,CLN-510064,PAT-31012,ENC-000044,2024-10-03,follow_up_note,LIC-VLN-RES-1044,FAC-AVL-201,Persistent Multisystem Follow-up,INTERVAL HISTORY\nThe patient returned after r...,confidential
4,CLN-510107,PAT-31115,ENC-000014,2021-06-08,clinical_summary,LIC-VLN-MED-1004,FAC-AVL-301,Clinical Summary 01,CONTEXT\nClinical summary for patient PAT-3111...,confidential


## **5/6 Clinical_observations**

In [39]:
df_clinical_observations.head()

,observation_number,encounter_number,patient_number,observation_time,observation_code,observation_name,numeric_value,text_value,unit,reference_range_text,interpretation_code,device_reference,quality_code,assessor_license_number
0,OBS-0000001,ENC-000001,PAT-31162,2018-02-27T09:27:00,RPULSE,Resting pulse,64.0,NaN,bpm,50–100,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
1,OBS-0000002,ENC-000001,PAT-31162,2018-02-27T09:27:30,APICAL,Apical cardiac rate,88.5,NaN,bpm,50–110,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
2,OBS-0000003,ENC-000001,PAT-31162,2018-02-27T09:28:00,PPULSE,Peripheral pulse rate,75.8,NaN,bpm,50–100,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
3,OBS-0000004,ENC-000001,PAT-31162,2018-02-27T09:28:30,CARDOUT,Estimated cardiac output,5.57,NaN,L/min,3.5–8.0,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
4,OBS-0000005,ENC-000001,PAT-31162,2018-02-27T09:29:00,O2SAT,Peripheral oxygen saturation,97.2,NaN,%,94–100,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001


In [40]:
df_clinical_observations.describe()


,observation_number,encounter_number,patient_number,observation_time,observation_code,observation_name,numeric_value,text_value,unit,reference_range_text,interpretation_code,device_reference,quality_code,assessor_license_number
count,17736,17736,17736,17736,17736,17736,15865,1871,15865,17736,17736,13410,17736,17736
unique,17734,1017,240,16451,32,56,2465,7,11,18,9,9,3,8
top,OBS-0003236,ENC-000006,PAT-31086,2025-06-20T13:27:00,RPULSE,Apical cardiac rate,100,normal,index,0–100,within_expected_range,DEV-006,acceptable,LIC-VLN-RES-1016
freq,2,32,102,17,1017,1016,183,730,7762,6831,12064,1574,17716,2575


In [41]:

# compte le nombre de valeurs uniques par colonne
unique_cols_clinical_Observations = []
for column in df_clinical_observations.columns:
    print(f'{column} : {df_clinical_observations[column].nunique()}')
    if df_clinical_observations[column].nunique() == 1:
        unique_cols_clinical_Observations.append(column)

print('---')
print(f'colonnes avec une seule valeur : {unique_cols_clinical_Observations}')

observation_number : 17734
encounter_number : 1017
patient_number : 240
observation_time : 16451
observation_code : 32
observation_name : 56
numeric_value : 2465
text_value : 7
unit : 11
reference_range_text : 18
interpretation_code : 9
device_reference : 9
quality_code : 3
assessor_license_number : 8
---
colonnes avec une seule valeur : []


In [42]:
print('Aucune valeur manquante')
df_clinical_observations.isna().sum()

Aucune valeur manquante


observation_number             0
encounter_number               0
patient_number                 0
observation_time               0
observation_code               0
observation_name               0
numeric_value               1871
text_value                 15865
unit                        1871
reference_range_text           0
interpretation_code            0
device_reference            4326
quality_code                   0
assessor_license_number        0
dtype: int64

In [43]:
print('Aucun int caché')
get_dtypes_report(df_clinical_observations)

Aucun int caché


,dtype,looks_numeric
observation_number,str,False
encounter_number,str,False
patient_number,str,False
observation_time,str,False
observation_code,str,False
observation_name,str,False
numeric_value,str,False
text_value,str,False
unit,str,False
reference_range_text,str,False


## Nettoyage

- Suppression des doublons
- Conversion de `numeric_value` en nombre, y compris les valeurs avec virgule (`84,9`) ou avec unité collée (`82.2 index`)
- Harmonisation des dates
- Mise en minuscules des catégories (les identifiants et les codes de mesure restent inchangés)
- Suppression des espaces inutiles

`text_value` et `numeric_value` sont complémentaires : une mesure est soit numérique, soit textuelle (par exemple les codes `OCULAR`, `NEURO`). Les valeurs manquantes de l'une sont donc normales, et nous conservons les deux colonnes ainsi que toutes les lignes.

In [44]:
enc = df_clinical_observations.copy()

print(quality_report(enc, "clinical_observations"))

# observation_code (RPULSE, BIOAGE…) est un code court : on le garde en majuscules
text_cols = colonnes_categories(enc, exclure=["observation_code"])

df_clinical_observations_clean = (
    enc
    .pipe(strip_whitespace)
    .pipe(normalize_case, cols=text_cols, mode="lower")
    .pipe(parse_dates,
          cols=['observation_time'],
          formats=["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S",
                   "%d-%b-%Y %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%b-%Y"])
    .pipe(remove_exact_duplicates)
)

# Conversion des valeurs numériques (virgule décimale et unité collée)
avant = df_clinical_observations_clean["numeric_value"].notna().sum()
df_clinical_observations_clean["numeric_value"] = to_number(df_clinical_observations_clean["numeric_value"])
apres = df_clinical_observations_clean["numeric_value"].notna().sum()
print(f"Valeurs numériques : {avant} avant conversion, {apres} après (aucune perdue si les deux sont égaux)")

# Codes d'observation conservés : les mesures textuelles sont bien présentes
print("Nombre de codes d'observation :", df_clinical_observations_clean["observation_code"].nunique())

{'name': 'clinical_observations', 'rows': 17736, 'columns': 14, 'duplicate_rows': 2, 'cells_missing': 23933, 'pct_cells_missing': np.float64(9.64), 'cols_with_missing': ['numeric_value', 'text_value', 'unit', 'device_reference'], 'constant_cols': []}


remove_exact_duplicates : 2 ligne(s) supprimée(s)
Valeurs numériques : 15864 avant conversion, 15864 après (aucune perdue si les deux sont égaux)
Nombre de codes d'observation : 32


In [45]:
df_clinical_observations_clean.dtypes

observation_number                 string
encounter_number                   string
patient_number                     string
observation_time           datetime64[ns]
observation_code                   string
observation_name                   string
numeric_value                     Float64
text_value                         string
unit                               string
reference_range_text               string
interpretation_code                string
device_reference                   string
quality_code                       string
assessor_license_number            string
dtype: object

In [46]:
df_clinical_observations_clean.head()

,observation_number,encounter_number,patient_number,observation_time,observation_code,observation_name,numeric_value,text_value,unit,reference_range_text,interpretation_code,device_reference,quality_code,assessor_license_number
0,OBS-0000001,ENC-000001,PAT-31162,2018-02-27 09:27:00,RPULSE,resting pulse,64.0,<NA>,bpm,50–100,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
1,OBS-0000002,ENC-000001,PAT-31162,2018-02-27 09:27:30,APICAL,apical cardiac rate,88.5,<NA>,bpm,50–110,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
2,OBS-0000003,ENC-000001,PAT-31162,2018-02-27 09:28:00,PPULSE,peripheral pulse rate,75.8,<NA>,bpm,50–100,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
3,OBS-0000004,ENC-000001,PAT-31162,2018-02-27 09:28:30,CARDOUT,estimated cardiac output,5.57,<NA>,l/min,3.5–8.0,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001
4,OBS-0000005,ENC-000001,PAT-31162,2018-02-27 09:29:00,O2SAT,peripheral oxygen saturation,97.2,<NA>,%,94–100,within_expected_range,DEV-002,acceptable,LIC-VLN-MED-1001


## **6/6 Patient_administration**

In [47]:
df_patient_administration.head()

,patient_number,family_name,given_name,date_of_birth,registered_sex,home_region,occupation_text,registration_date,primary_facility_license,record_status
0,PAT-31162,Varen,Eli,1962-04-19,male,Valenne Republic,corporate adviser,2018-01-01,FAC-VLN-101,inactive
1,PAT-31040,Valen,Jonah,1972-06-12,male,Valenne Republic,research professional,2017-02-01,FAC-AVL-201,active
2,PAT-31115,Oren,Nadia,1979-09-03,female,Valenne Republic,clinical specialist,2018-05-01,FAC-AVL-301,active
3,PAT-31186,Kader,Soren,1969-02-11,male,Nordhaven Union,corporate adviser,2017-06-01,FAC-VLN-101,active
4,PAT-31215,Roe,Ilan,1976-07-07,male,Avelora Federation,research professional,2018-05-01,FAC-AVL-201,active


## Stats

In [48]:
df_patient_administration.describe()

,patient_number,family_name,given_name,date_of_birth,registered_sex,home_region,occupation_text,registration_date,primary_facility_license,record_status
count,241,241,241,241,241,241,241,241,241,241
unique,240,67,67,239,3,6,46,65,3,4
top,PAT-31045,Garen,Taren,1976-07-07,female,Avelora Federation,retired professional,2021-09-01,FAC-VLN-101,active
freq,2,9,10,2,97,54,29,8,151,230


In [49]:

# compte le nombre de valeurs uniques par colonne
unique_cols_patient_administrations = []
for column in df_patient_administration.columns:
    print(f'{column} : {df_patient_administration[column].nunique()}')
    if df_patient_administration[column].nunique() == 1:
        unique_cols_patient_administrations.append(column)

print('---')
print(f'colonnes avec une seule valeur : {unique_cols_patient_administrations}')

patient_number : 240
family_name : 67
given_name : 67
date_of_birth : 239
registered_sex : 3
home_region : 6
occupation_text : 46
registration_date : 65
primary_facility_license : 3
record_status : 4
---
colonnes avec une seule valeur : []


In [50]:
print('Aucune valeur manquante')
df_patient_administration.isna().sum()

Aucune valeur manquante


patient_number              0
family_name                 0
given_name                  0
date_of_birth               0
registered_sex              0
home_region                 0
occupation_text             0
registration_date           0
primary_facility_license    0
record_status               0
dtype: int64

In [51]:

print('Aucun int caché')
get_dtypes_report(df_patient_administration)

Aucun int caché


,dtype,looks_numeric
patient_number,str,False
family_name,str,False
given_name,str,False
date_of_birth,str,False
registered_sex,str,False
home_region,str,False
occupation_text,str,False
registration_date,str,False
primary_facility_license,str,False
record_status,str,False


## Nettoyage

- Harmonisation des dates
- Mise en minuscules des catégories (sexe, région, profession, statut)
- Suppression des espaces dans les textes

Les identifiants et les noms de personnes gardent leur écriture d'origine, pour pouvoir les rapprocher des autres domaines.

In [52]:
enc = df_patient_administration.copy()

print(quality_report(enc, "patient_administration"))

text_cols = colonnes_categories(enc, exclure=["family_name", "given_name"])

df_patient_administration_clean = (
    enc
    .pipe(strip_whitespace)
    .pipe(normalize_case, cols=text_cols, mode="lower")
    .pipe(parse_dates,
          cols=['date_of_birth', 'registration_date'],
          formats=["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S",
                   "%d-%b-%Y %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%b-%Y"])
    .pipe(remove_exact_duplicates)
)

check_unique_key(df_patient_administration_clean, "patient_number")

{'name': 'patient_administration', 'rows': 241, 'columns': 10, 'duplicate_rows': 1, 'cells_missing': 0, 'pct_cells_missing': np.float64(0.0), 'cols_with_missing': [], 'constant_cols': []}
remove_exact_duplicates : 1 ligne(s) supprimée(s)


,patient_number,family_name,given_name,date_of_birth,registered_sex,home_region,occupation_text,registration_date,primary_facility_license,record_status


In [53]:
df_patient_administration_clean.head()

,patient_number,family_name,given_name,date_of_birth,registered_sex,home_region,occupation_text,registration_date,primary_facility_license,record_status
0,PAT-31162,Varen,Eli,1962-04-19,male,valenne republic,corporate adviser,2018-01-01,FAC-VLN-101,inactive
1,PAT-31040,Valen,Jonah,1972-06-12,male,valenne republic,research professional,2017-02-01,FAC-AVL-201,active
2,PAT-31115,Oren,Nadia,1979-09-03,female,valenne republic,clinical specialist,2018-05-01,FAC-AVL-301,active
3,PAT-31186,Kader,Soren,1969-02-11,male,nordhaven union,corporate adviser,2017-06-01,FAC-VLN-101,active
4,PAT-31215,Roe,Ilan,1976-07-07,male,avelora federation,research professional,2018-05-01,FAC-AVL-201,active


In [54]:
df_patient_administration_clean.dtypes

patient_number                      string
family_name                         string
given_name                          string
date_of_birth               datetime64[ns]
registered_sex                      string
home_region                         string
occupation_text                     string
registration_date           datetime64[ns]
primary_facility_license            string
record_status                       string
dtype: object

## **7/7 Clinical_procedures**

Ce fichier n'avait pas été nettoyé. Il contient les actes médicaux, dont la procédure `MRP-42` (« Targeted metabolic resilience administration »).

In [55]:
df_clinical_procedures.head()

,procedure_record_id,encounter_number,patient_number,procedure_code,procedure_description,scheduled_time,performed_time,performing_facility_license,performing_license_number,authorization_class,material_reference,procedure_status,outcome_code,next_review_date
0,PRC-000001,ENC-001018,PAT-31221,MRP-42,Targeted metabolic resilience administration,2016-02-18T08:30:00,2016-02-18T09:05:00,FAC-VLN-101,LIC-VLN-MED-1002,restricted_specialist,MTL-2016-0002,completed,completed_without_immediate_complication,2016-03-22
1,PRC-000002,ENC-001019,PAT-31221,MRP-42,Targeted metabolic resilience administration,2018-02-22T08:30:00,2018-02-22T09:05:00,FAC-VLN-101,LIC-VLN-MED-1002,restricted_specialist,MTL-2018-0003,completed,completed_without_immediate_complication,2018-04-01
2,PRC-000003,ENC-001020,PAT-31221,MRP-42,Targeted metabolic resilience administration,2020-03-05T08:30:00,2020-03-05T09:05:00,FAC-VLN-101,LIC-VLN-MED-1002,restricted_specialist,MTL-2020-0004,completed,completed_without_immediate_complication,2020-04-17
3,PRC-000004,ENC-001021,PAT-31221,MRP-42,Targeted metabolic resilience administration,2022-02-17T08:30:00,2022-02-17T09:05:00,FAC-VLN-101,LIC-VLN-MED-1002,restricted_specialist,MTL-2022-0005,completed,completed_without_immediate_complication,2022-03-17
4,PRC-000005,ENC-001022,PAT-31221,MRP-42,Targeted metabolic resilience administration,2024-03-01T08:30:00,2024-03-01T09:05:00,FAC-VLN-101,LIC-VLN-MED-1002,restricted_specialist,MTL-2024-0006,completed,completed_without_immediate_complication,2024-04-03


In [56]:
print(quality_report(df_clinical_procedures, "clinical_procedures"))
df_clinical_procedures.isna().sum()

{'name': 'clinical_procedures', 'rows': 311, 'columns': 14, 'duplicate_rows': 1, 'cells_missing': 317, 'pct_cells_missing': np.float64(7.28), 'cols_with_missing': ['material_reference', 'next_review_date'], 'constant_cols': ['procedure_status']}


procedure_record_id              0
encounter_number                 0
patient_number                   0
procedure_code                   0
procedure_description            0
scheduled_time                   0
performed_time                   0
performing_facility_license      0
performing_license_number        0
authorization_class              0
material_reference             261
procedure_status                 0
outcome_code                     0
next_review_date                56
dtype: int64

## Nettoyage

- Suppression des doublons
- Harmonisation des dates
- Mise en minuscules des catégories (les codes et identifiants restent inchangés)

`material_reference` est vide pour la plupart des actes : seuls certains actes utilisent un matériel. Une case vide signifie « aucun matériel », nous la conservons.

In [57]:
enc = df_clinical_procedures.copy()

# procedure_code (MRP-42, OCT-TRK…) est un code court : on le garde en majuscules
text_cols = colonnes_categories(enc, exclure=["procedure_code"])

df_clinical_procedures_clean = (
    enc
    .pipe(strip_whitespace)
    .pipe(normalize_case, cols=text_cols, mode="lower")
    .pipe(parse_dates,
          cols=['scheduled_time', 'performed_time', 'next_review_date'],
          formats=["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S",
                   "%d-%b-%Y %H:%M:%S", "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%d-%b-%Y"])
    .pipe(remove_exact_duplicates)
)

check_unique_key(df_clinical_procedures_clean, "procedure_record_id")
df_clinical_procedures_clean["procedure_code"].value_counts()

remove_exact_duplicates : 1 ligne(s) supprimée(s)


procedure_code
OCT-TRK     51
MRP-42      50
OCC-PNL     49
AUT-REV     43
CARD-IMG    41
AGE-PNL     39
MOB-PNL     37
Name: count, dtype: int64[pyarrow]

## **Sauvegarde des fichiers nettoyés**

In [58]:
import pyarrow as pa, pandas as pd


In [59]:
# Les tables nettoyées sont enregistrées dans data_clean/health/,
# avec le nom du fichier d'origine (même convention que les autres domaines).
# Le dossier data/ du prof n'est jamais modifié.
OUT_DIR = Path("data_clean/health")
OUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "cardiac_recordings": df_cardiac_recording_clean,
    "cardiac_signal_samples": df_cardiac_signal_samples_clean,
    "clinical_encounters": df_clinical_encounters_clean,
    "clinical_notes": df_cardiac_notes_clean,
    "clinical_observations": df_clinical_observations_clean,
    "patient_administration": df_patient_administration_clean,
    "clinical_procedures": df_clinical_procedures_clean,
}

for name, df in datasets.items():
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"{name}.csv : {len(df)} lignes, {df.shape[1]} colonnes")

cardiac_recordings.csv : 44 lignes, 16 colonnes


cardiac_signal_samples.csv : 96925 lignes, 4 colonnes


clinical_encounters.csv : 1067 lignes, 12 colonnes
clinical_notes.csv : 30 lignes, 10 colonnes


clinical_observations.csv : 17734 lignes, 14 colonnes
patient_administration.csv : 240 lignes, 10 colonnes
clinical_procedures.csv : 310 lignes, 14 colonnes
